Libraries and Data

In [52]:
import medspacy
import pandas as pd
import re
from pathlib import Path
from medspacy.ner import TargetRule

ROOT = Path('../../data/raw')
data = ROOT / 'cases.csv'
metadata = ROOT / 'metadata.csv'


Visualizing

In [53]:
data = pd.read_csv(data)
metadata = pd.read_csv(metadata)

data.head()
metadata.head()


,article_id,authors,case_amount,doi,journal,journal_detail,keywords,license,link,major_mesh_terms,mesh_terms,pmcid,pmid,title,year
0,PMC5137649,"[C E Bailey, M B Fritz, L Webb, N B Merchant, ...",1,10.1308/003588414X13824511649977,Ann R Coll Surg Engl,2014 Jan;96(1):88E-90E.,NaN,CC BY,https://pubmed.ncbi.nlm.nih.gov/24417851/,"[Cysts / diagnosis, Stomach / abnormalities, S...","[Cysts / diagnosis, Stomach / abnormalities, S...",PMC5137649,24417851,Gastric duplication cyst masquerading as a muc...,2014
1,PMC9387390,"[Elias A Chamely, Bryan Hoang, Nadim S Jafri, ...",1,10.4293/CRSLS.2021.00094,CRSLS,2022 Feb 25;9(1):e2021.00094.,"[delayed gastric emptying, endoscopy, gastric ...",CC BY-NC-SA,https://pubmed.ncbi.nlm.nih.gov/36016812/,"[Adenocarcinoma / complications, Gastric Bypas...","[Adenocarcinoma / complications, Gastric Bypas...",PMC9387390,36016812,Palliative Endoscopic Salvage of a Functionall...,2022
2,PMC3437073,[M Y Al-Naami],1,NaN,J Family Community Med,1999 Jan;6(1):45-8.,"[abscess, spleen, tuberculous]",CC BY-NC-SA,https://pubmed.ncbi.nlm.nih.gov/23008596/,[],[Case Reports],PMC3437073,23008596,An unusual presentation of tuberculous splenic...,1999
3,PMC7102447,"[Zeid Nesheiwat, Pinang Shastri, Rohit Vyas, C...",1,10.1155/2020/7842591,Case Rep Cardiol,2020 Jan 11;2020:7842591.,NaN,CC BY,https://pubmed.ncbi.nlm.nih.gov/32257451/,[],[Case Reports],PMC7102447,32257451,A Case of Acute Massive Bioprosthetic Mitral V...,2020
4,PMC6354154,"[Patrícia Alves, Inês Sá, Miguel Brito, Cátia ...",1,10.1155/2019/2537480,Case Rep Obstet Gynecol,2019 Jan 16;2019:2537480.,NaN,CC BY,https://pubmed.ncbi.nlm.nih.gov/30792930/,[],[Case Reports],PMC6354154,30792930,An Early Diagnosis of an Ovarian Steroid Cell ...,2019


inner joining important mesh terms

In [54]:
full_data = pd.merge(data, metadata)
full_data = full_data[['case_text','gender', 'case_id', 'major_mesh_terms']]
full_data.head()

,case_text,gender,case_id,major_mesh_terms
0,A 44-year-old woman presented with a 3-day his...,Female,PMC5137649_01,"[Cysts / diagnosis, Stomach / abnormalities, S..."
1,A 57-year-old man with no significant past med...,Male,PMC9387390_01,"[Adenocarcinoma / complications, Gastric Bypas..."
2,A 55-year-old male presented with a gradually ...,Male,PMC3437073_01,[]
3,A 65-year-old male with a past medical history...,Male,PMC7102447_01,[]
4,A 30-year-old nulligravida presented herself i...,Female,PMC6354154_01,[]


picking the first case_text

In [55]:
text = full_data['case_text'][0]
print(text)

A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A laparoscopic distal panc

setting dictionary

In [56]:
dictionary = {
    # 1. Regiões Anatômicas e Estruturas Comuns
    "anatomy": [
        "heart",
        "lung",
        "lungs",
        "stomach",
        "liver",
        "kidney",
        "kidneys",
        "brain",
        "abdomen",
        "chest",
        "spine",
        "skin",
        "blood vessels",
        "artery",
        "vein",
        "bowel",
        "intestine",
        "colon",
        "pancreas",
        "esophagus",
        "pelvis",
        "neck",
        "head",
        "bone",
        "joint",
    ],
    # 2. Sintomas e Queixas Mais Recorrentes
    "symptom": [
        "pain",
        "chest pain",
        "abdominal pain",
        "headache",
        "fever",
        "cough",
        "shortness of breath",
        "dyspnea",
        "nausea",
        "vomiting",
        "fatigue",
        "dizziness",
        "diarrhea",
        "constipation",
        "edema",
        "swelling",
        "weight loss",
        "chills",
        "sweating",
        "palpitations",
        "weakness",
        "bleeding",
    ],
    # 3. Diagnósticos e Condições Crônicas/Agudas Frequentes
    "diagnosis": [
        "hypertension",
        "diabetes mellitus",
        "asthma",
        "pneumonia",
        "myocardial infarction",
        "stroke",
        "heart failure",
        "renal failure",
        "cirrhosis",
        "anemia",
        "sepsis",
        "cancer",
        "tumor",
        "infection",
        "embolism",
        "arrhythmia",
        "atrial fibrillation",
        "depression",
        "anxiety",
    ],
    # 4. Tratamentos e Procedimentos Padrão
    "treatment": [
        "surgery",
        "resection",
        "transplant",
        "chemotherapy",
        "radiation therapy",
        "dialysis",
        "intubation",
        "mechanical ventilation",
        "blood transfusion",
        "biopsy",
        "catheterization",
        "drainage",
        "physical therapy",
        "immunization",
    ],
    # 5. Medicamentos Mais Prescritos (Classes e Nomes Comuns)
    "medication": [
        "aspirin",
        "paracetamol",
        "ibuprofen",
        "insulin",
        "metformin",
        "omeprazole",
        "furosemide",
        "heparin",
        "warfarin",
        "atorvastatin",
        "enalapril",
        "amoxicillin",
        "morphine",
        "prednisone",
        "antibiotic",
        "analgesic",
    ],
    # 6. Exames Laboratoriais e de Imagem Padrão
    "exam": [
        "blood test",
        "complete blood count",
        "cbc",
        "electrocardiogram",
        "ecg",
        "ekg",
        "chest X-ray",
        "computed tomography",
        "ct scan",
        "magnetic resonance imaging",
        "mri",
        "ultrasound",
        "urinalysis",
        "echocardiogram",
        "biopsy report",
        "endoscopy",
    ],
}

In [57]:
nlp = medspacy.load()

# 2. Pega o componente TargetMatcher que já vem no pipeline
target_matcher = nlp.get_pipe('medspacy_target_matcher')

rules = []
for category, terms in dictionary.items():
  for term in terms:
    rules.append(TargetRule(literal=term, category=category.upper()))

target_matcher.add(rules)

processed_text = nlp(text)

for ent in processed_text.ents:
  print(f"Entidade: {ent.text} | Classe (Tipo para o Grafo): {ent.label_}")

2026-09-08 22:56:04.823 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 0 'A' marked as sentence start (span begin)
2026-09-08 22:56:04.824 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 28 'Her' marked as sentence start (span end next token)
2026-09-08 22:56:04.824 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 28 'Her' marked as sentence start (span begin)
2026-09-08 22:56:04.824 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 48 'She' marked as sentence start (span end next token)
2026-09-08 22:56:04.825 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 48 'She' marked as sentence start (span begin)
2026-09-08 22:56:04.825 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 76 'She' marked as

Entidade: abdominal pain | Classe (Tipo para o Grafo): SYMPTOM
Entidade: nausea | Classe (Tipo para o Grafo): SYMPTOM
Entidade: constipation | Classe (Tipo para o Grafo): SYMPTOM
Entidade: computed tomography | Classe (Tipo para o Grafo): EXAM
Entidade: stomach | Classe (Tipo para o Grafo): ANATOMY
Entidade: pancreas | Classe (Tipo para o Grafo): ANATOMY
Entidade: stomach | Classe (Tipo para o Grafo): ANATOMY
Entidade: resection | Classe (Tipo para o Grafo): TREATMENT
Entidade: stomach | Classe (Tipo para o Grafo): ANATOMY
Entidade: stomach | Classe (Tipo para o Grafo): ANATOMY
Entidade: pancreas | Classe (Tipo para o Grafo): ANATOMY
Entidade: pancreas | Classe (Tipo para o Grafo): ANATOMY
Entidade: stomach | Classe (Tipo para o Grafo): ANATOMY
Entidade: endoscopy | Classe (Tipo para o Grafo): EXAM
Entidade: stomach | Classe (Tipo para o Grafo): ANATOMY
Entidade: resection | Classe (Tipo para o Grafo): TREATMENT
Entidade: stomach | Classe (Tipo para o Grafo): ANATOMY
Entidade: resectio

In [58]:
without_stopwords_text = [token for token in processed_text if not token.is_stop and not token.is_punct and not '\n' in token.text]